# 03 — Modeling: Q5 (Secondary Target)
## *Does society put unhealthy or bad pressure on men?*

**Why this target?**  
Q5 measures a *social/political belief* about structural gender pressure rather than a personal behavior. It complements Q17 by asking whether men perceive the system itself as burdensome.

**Class balance:** Yes = 955 (60%) / No = 647 (40%).

**Note on performance:** Q5 is harder to predict than Q17 (~0.63 AUC vs ~0.66). Political and social beliefs are noisier targets.

**Q7 column mapping (per survey instrument):**
- q0007_0001: Ask a friend for professional advice
- q0007_0002: Ask a friend for personal advice
- q0007_0003: Express physical affection to male friends
- q0007_0004: Cry
- q0007_0005: Get in a physical fight
- q0007_0006: Have sexual relations with women
- q0007_0007: Have sexual relations with men
- q0007_0008: Watch sports
- q0007_0009: Work out
- q0007_0010: See a therapist
- q0007_0011: Feel lonely or isolated

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, RocCurveDisplay, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

raw = pd.read_csv('../data/raw-responses.csv')
df_clean = pd.read_csv('../data/cleaned-responses.csv')

## 1. Feature Engineering

In [ ]:
def build_features(raw, df_clean, exclude_col=None):
    """Assemble the full feature matrix. See notebook 02 for full documentation."""
    q4_cols  = [c for c in df_clean.columns if c.startswith('q0004')]
    # Q7 ordinal encoding — correct mapping per survey instrument
    q7_map   = {'Often': 4, 'Sometimes': 3, 'Rarely': 2,
                'Never, but open to it': 1, 'Never, and not open to it': 0, 'No answer': np.nan}
    q7_cols  = [c for c in raw.columns if c.startswith('q0007')]
    q7_df    = raw[q7_cols].replace(q7_map)
    q8_cols  = [c for c in df_clean.columns if c.startswith('q0008')]
    q9       = df_clean['q0009'].map({'employed': 1, 'not_employed': 0})
    q10_cols = [c for c in df_clean.columns if c.startswith('q0010')]
    q11_cols = [c for c in df_clean.columns if c.startswith('q0011')]
    q12_cols = [c for c in df_clean.columns if c.startswith('q0012')]
    q14      = df_clean['auto_q0014'].replace({0: np.nan})
    q17      = raw['q0017'].map({'Yes': 1, 'No': 0})
    q18_map  = {'Always': 5, 'Often': 4, 'Sometimes': 3, 'Rarely': 2, 'Never': 1, 'No answer': np.nan}
    q18      = raw['q0018'].map(q18_map)
    q19_cols = [c for c in df_clean.columns if c.startswith('q0019')]
    q20_cols = [c for c in df_clean.columns if c.startswith('q0020')]
    q21_cols = [c for c in df_clean.columns if c.startswith('q0021')]
    q22      = df_clean['auto_q0022'].map({0: 0, 1: 1, 2: np.nan})

    feat = pd.concat([
        df_clean[q4_cols], q7_df.reset_index(drop=True), df_clean[q8_cols],
        q9.rename('q0009'), df_clean[q10_cols], df_clean[q11_cols], df_clean[q12_cols],
        q14.rename('q0014'), q17.reset_index(drop=True).rename('q0017'),
        q18.reset_index(drop=True).rename('q0018'),
        df_clean[q19_cols], df_clean[q20_cols], df_clean[q21_cols],
        q22.rename('q0022'),
        df_clean['auto_q0024'].rename('marital'),
        df_clean['auto_orientation'].rename('orientation'),
        df_clean['auto_race2'].rename('race'),
        df_clean['auto_educ4'].rename('educ'),
        df_clean['auto_age3'].rename('age'),
        df_clean['auto_kids'].map({0: 0, 1: 1, 2: np.nan}).rename('kids'),
    ], axis=1)

    if exclude_col and exclude_col in feat.columns:
        feat = feat.drop(columns=[exclude_col])
    return feat


features = build_features(raw, df_clean, exclude_col=None)
y = raw['q0005'].map({'Yes': 1, 'No': 0}).reset_index(drop=True)
mask = y.notna()
X_raw = features[mask].copy()
y = y[mask].copy()

imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns)

print(f"Feature matrix: {X.shape[0]} rows x {X.shape[1]} features")
print(f"Target: Yes={int((y==1).sum())} ({(y==1).mean()*100:.1f}%), No={int((y==0).sum())} ({(y==0).mean()*100:.1f}%)")
print(f"Majority class baseline: {y.value_counts(normalize=True).max():.3f}")

## 2. EDA: Key Feature Distributions by Target

In [ ]:
q7_cols = [c for c in raw.columns if c.startswith('q0007')]
q7_map = {'Often': 4, 'Sometimes': 3, 'Rarely': 2,
          'Never, but open to it': 1, 'Never, and not open to it': 0, 'No answer': np.nan}

# Correct labels per survey instrument
q7_labels_short = [
    'Ask friend\n(professional)', 'Ask friend\n(personal)', 'Physical\naffection',
    'Cry', 'Physical\nfight', 'Sex\n(women)', 'Sex\n(men)', 'Watch\nsports',
    'Work out', 'See\ntherapist', 'Feel\nlonely'
]

plot_df = raw[q7_cols].replace(q7_map).copy()
plot_df['target'] = raw['q0005'].map({'Yes': 'Yes', 'No': 'No'})
plot_df = plot_df.dropna(subset=['target'])

yes_means = plot_df[plot_df['target']=='Yes'][q7_cols].mean()
no_means  = plot_df[plot_df['target']=='No'][q7_cols].mean()

x = np.arange(len(q7_cols))
width = 0.38
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, yes_means.values, width, label='Yes (society pressures men)', color='#388E3C', alpha=0.85)
ax.bar(x + width/2, no_means.values,  width, label='No',                          color='#FF5722', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(q7_labels_short, fontsize=8)
ax.set_ylabel('Mean frequency (0=Never closed, 4=Often)')
ax.set_title('Q7: Lifestyle Frequencies by Q5 Target', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 4.8)
plt.tight_layout()
plt.show()

In [ ]:
q8_labels = {
    'q0008_0001': 'Height', 'q0008_0002': 'Weight', 'q0008_0003': 'Hair/hairline',
    'q0008_0004': 'Physique', 'q0008_0005': 'Genitalia', 'q0008_0006': 'Clothing/style',
    'q0008_0007': 'Sexual performance', 'q0008_0008': 'Mental health',
    'q0008_0009': 'Physical health', 'q0008_0010': 'Finances',
    'q0008_0011': 'Ability to provide', 'q0008_0012': 'None of above',
}
q8_cols = [c for c in df_clean.columns if c.startswith('q0008')]
q8_plot = df_clean[q8_cols].copy()
q8_plot['target'] = raw['q0005'].map({'Yes': 'Yes', 'No': 'No'}).values
q8_plot = q8_plot.dropna(subset=['target'])

diff = (q8_plot[q8_plot['target']=='Yes'][q8_cols].mean() -
        q8_plot[q8_plot['target']=='No'][q8_cols].mean())
diff.index = [q8_labels[c] for c in diff.index]
diff = diff.sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#388E3C' if d > 0 else '#FF5722' for d in diff]
ax.barh(diff.index, diff.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Q8: Worry Rate Difference (Yes - No)\nGreen = more common among those who think society pressures men',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Model Training & Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=1000, C=1.0))
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42
    ),
}

results = {}
majority_baseline = y.value_counts(normalize=True).max()

print(f"Majority class baseline: {majority_baseline:.3f}\n")
print(f"{'Model':<25} {'Accuracy':>12} {'AUC':>12}")
print('-' * 51)

for name, model in models.items():
    acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    results[name] = dict(acc_mean=acc.mean(), acc_std=acc.std(),
                         auc_mean=auc.mean(), auc_std=auc.std())
    print(f"{name:<25} {acc.mean():.3f}±{acc.std():.3f}  {auc.mean():.3f}±{auc.std():.3f}")

## 4. ROC Curves & Confusion Matrix

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#1976D2', '#388E3C', '#F57C00']
for (name, model), color in zip(models.items(), colors):
    model.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[0], name=name, color=color)
axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random baseline')
axes[0].set_title('ROC Curves — Q5', fontsize=12, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=9)

rf = models['Random Forest']
y_pred = rf.predict(X_test)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['No pressure', 'Pressure exists'],
    ax=axes[1], colorbar=False, cmap='Greens'
)
axes[1].set_title('Confusion Matrix — Random Forest', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nRandom Forest — Classification Report:')
print(classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)']))

## 5. Feature Importance

In [ ]:
# Correct Q7 labels per survey instrument
feat_labels = {
    'q0018':      'Q18: Tries to pay on dates',
    'q0017':      'Q17: Feels expected to make first move',
    'q0007_0011': 'Q7: Feels lonely/isolated',
    'q0007_0010': 'Q7: Sees therapist',
    'q0007_0009': 'Q7: Works out',
    'q0007_0008': 'Q7: Watches sports',
    'q0007_0007': 'Q7: Sex with men',
    'q0007_0006': 'Q7: Sex with women',
    'q0007_0005': 'Q7: Gets in a physical fight',
    'q0007_0004': 'Q7: Cries',
    'q0007_0003': 'Q7: Physical affection — male friends',
    'q0007_0002': 'Q7: Asks friend for personal advice',
    'q0007_0001': 'Q7: Asks friend for professional advice',
    'q0004_0001': 'Q4: Masculinity ideas from father',
    'q0004_0004': 'Q4: Masculinity ideas from pop culture',
    'q0004_0003': 'Q4: Masculinity ideas from other family',
    'q0008_0010': 'Q8: Worries about finances',
    'q0008_0008': 'Q8: Worries about mental health',
    'q0008_0007': 'Q8: Worries about sexual performance',
    'q0008_0002': 'Q8: Worries about weight',
    'q0011_0002': 'Q11: Disadvantage — harassment accusation risk',
    'q0011_0003': 'Q11: Disadvantage — sexism accusation risk',
    'q0011_0005': 'Q11: Disadvantage — none of the above',
    'q0010_0005': 'Q10: Advantage — men praised more',
    'q0012_0005': 'Q12: Harassment response — did nothing',
    'q0012_0006': 'Q12: Never witnessed harassment',
    'q0021_0004': 'Q21: None of the above (sexual boundary)',
    'q0019_0006': 'Q19: Pays — to see if other offers to share',
    'orientation':'Demographics: Sexual orientation',
    'educ':       'Demographics: Education',
    'age':        'Demographics: Age group',
    'q0009':      'Q9: Employment status',
}

rf_fit = models['Random Forest']
perm = permutation_importance(rf_fit, X_test, y_test, n_repeats=15, random_state=42, n_jobs=-1)
fi = pd.DataFrame({
    'feature': X.columns,
    'importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('importance', ascending=False).head(20)
fi['label'] = fi['feature'].map(feat_labels).fillna(fi['feature'])

fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = ['#388E3C' if v > 0 else '#E0E0E0' for v in fi['importance']]
ax.barh(fi['label'][::-1], fi['importance'][::-1],
        xerr=fi['std'][::-1], color=colors_bar[::-1], capsize=3, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.6)
ax.set_xlabel('Permutation importance (mean accuracy drop)')
ax.set_title('Top 20 Features — Q5\n(Random Forest, permutation importance)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
lr_pipe = models['Logistic Regression']
lr_pipe.fit(X_train, y_train)
lr_coef = pd.DataFrame({
    'feature': X.columns,
    'coef': lr_pipe.named_steps['lr'].coef_[0]
}).sort_values('coef', key=abs, ascending=False).head(20)
lr_coef['label'] = lr_coef['feature'].map(feat_labels).fillna(lr_coef['feature'])

fig, ax = plt.subplots(figsize=(10, 8))
colors_lr = ['#388E3C' if c > 0 else '#FF5722' for c in lr_coef['coef']]
ax.barh(lr_coef['label'][::-1], lr_coef['coef'][::-1], color=colors_lr[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized coefficient')
ax.set_title('Logistic Regression Coefficients — Q5\nGreen = predicts Yes, Red = predicts No',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Key Findings

**Model performance:** ~62% accuracy / ~0.63 AUC. Weaker than Q17 because behavioral survey features capture enacted norms better than abstract beliefs about social structure.

**Predicts believing society pressures men (Yes):**
- **Feels lonely/isolated frequently** — emotional burden as evidence of systemic pressure
- **Sees a therapist** — self-aware about the cost of masculine expectations
- **Worries about finances and sexual performance** — experiencing the specific pressures the question asks about
- **Feels expected to make first move (Q17)** — internalized norms and perceived pressure travel together
- **Masculinity ideas from pop culture** — absorbing scripted norms may increase awareness of their weight

**Predicts NOT believing society pressures men (No):**
- **Non-straight orientation** — less subject to the heteronormative pressure being asked about
- **Works out frequently** — comfortable performing traditional masculinity
- **Never witnessed harassment** — generally non-reflective about gender dynamics
- **Masculinity ideas from other family members** — more diffuse, less scripted source